# EXPERIMENT: cluster price feature (2-GPU). NOT part of the report

**This notebook is a separate experiment.** It is not `housing_part1.ipynb` (the report notebook) and its results must not be mixed with it.

It tests one extra feature, the **cluster price**: k-means on latitude/longitude (as in the report), then **one column** with the average training price of the district's cluster. Because this feature is built from the response variable, check with the instructor that it is allowed before submitting anything from here.

How the feature avoids leakage:

* **Training rows:** out-of-fold. The training rows are split into 5 parts and each part gets cluster averages computed from the other 4, so a district never uses its own price.
* **Other rows** (early stopping, validation fold, test set): averages from all the training rows of that fit.
* **Smoothing:** each cluster average is pulled towards the overall average price, `m` = how many districts' worth of overall average is mixed in (`n / (n + m)` of a cluster with `n` districts is its own average). This matters for small clusters, whose average would otherwise be noise.
* Everything is inside the pipelines, so every CV fold recomputes it from that fold's training rows only.

**Steps:** (1) tune `n_clusters` × `m`: ridge on the full grid (context), **NN on the full grid (decides)**, plus a "cluster price + the 1200 similarity columns" check; (2) full NN retuning on the chosen feature set; (3) recheck of the top pairs with the retuned configuration; (4) comparison with the current best model (NN, k-means 1200/500) on the same folds, and submissions.

All outputs go to `OUTPUT_DIR = .../outputs_EXPERIMENT_cluster_price`, submissions are called `submission_EXP_...csv`. The networks are trained two at a time, one per GPU (see `housing_part1_2gpu.ipynb`).

# Packages

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as path_effects
import seaborn as sns
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from scipy.spatial.distance import cdist
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, FunctionTransformer, OneHotEncoder
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, RidgeCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_validate
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.metrics.pairwise import rbf_kernel
import itertools
import hashlib
from sklearn.base import BaseEstimator, TransformerMixin
import time
from tqdm.auto import tqdm

# For NN: Use CUDA if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using acceleration device: {device}")

# General settings and data import

In [ ]:
SEED=118

#from google.colab import drive # Only if running from Colab
#drive.mount('/content/drive') # Only if running from Colab
DATA_DIR = "/kaggle/working" # Change to appropriate local path
OUTPUT_DIR = os.path.join(DATA_DIR, "outputs_EXPERIMENT_cluster_price") # separate from the report outputs
os.makedirs(OUTPUT_DIR, exist_ok=True) # If the folder doesn't exist, create it

x_train_df = pd.read_csv("/kaggle/input/datasets/eduardovitale/california-housing-kaggle-comp/x_train_houses.csv")
y_train_df = pd.read_csv("/kaggle/input/datasets/eduardovitale/california-housing-kaggle-comp/y_train_houses.csv")
x_test_df = pd.read_csv("/kaggle/input/datasets/eduardovitale/california-housing-kaggle-comp/x_test_houses.csv")

features=x_train_df.columns[1:].tolist()
y_tr_raw=y_train_df.iloc[:, 1].values

# IDA and EDA

In [ ]:
x_train = x_train_df[features]
x_test = x_test_df[features]

# 1. Look for missing data
print(x_train.isna().sum())
print(x_test.isna().sum()) # there is missing data in total_bedrooms

# 2. Numerical variables' distributions
print(x_train.describe())
x_train.assign(Price=y_tr_raw).hist(bins=40, figsize=(12,8))
plt.tight_layout()
plt.show()

# 3. Plot house on a map based on latitude/longitude
cities=np.array([[34.1141,-118.4068],   # Los Angeles
                 [38.5677,-121.4685],   # Sacramento
                 [32.8313,-117.1222],   # San Diego
                 [37.7558,-122.4449],   # San Francisco
                 [37.3012,-121.8480]])  # San Jose
# source: https://simplemaps.com/data/us-cities
city_names = ["Los Angeles", "Sacramento", "San Diego", "San Francisco", "San Jose"]

plt.figure(figsize=(8,8))
plt.scatter(x_train["longitude"], x_train["latitude"], c=y_tr_raw, alpha=0.4, s=7)
plt.colorbar(label="Price")
plt.scatter(cities[:, 1], cities[:, 0], color="red", edgecolor="white", s=150, marker="*")
for i, name in enumerate(city_names):
    txt = plt.text(cities[i, 1] + 0.15, cities[i, 0] - 0.05, name, color="black", fontsize=10, fontweight="bold")
    txt.set_path_effects([path_effects.withStroke(linewidth=3, foreground="white")])
plt.gca().set_aspect("equal")
plt.xlabel("longitude")
plt.ylabel("latitude")
plt.show()
# Expensive houses near the big cities. An engineered feature that measures distance from them could have improving effects?

# 4. Mean price by Ocean proximity
sns.boxplot(data=x_train.assign(Price=y_tr_raw), x="ocean_proximity", y="Price",
            order=x_train.assign(Price=y_tr_raw).groupby("ocean_proximity")["Price"].mean().sort_values(ascending=False).index)
plt.show()
# Expensive houses are located near the coast

# 5. Correlations
heatmap = sns.heatmap(x_train.assign(Price=y_tr_raw).select_dtypes("number").corr(), cmap="coolwarm", annot=True)
## Correlations with response variable "Price"
corr_price = x_train.assign(Price=y_tr_raw).select_dtypes("number").corr()["Price"].sort_values(key=abs, ascending=False)
corr_price
# most correlated variable: median_income

# Feature engineering

In [ ]:
# Histograms: the totals measure district size -> per-household ratios describe the typical home
def ratios(d):
    return d.assign(rooms_per_household=d["total_rooms"]/d["households"],
                    population_per_household=d["population"]/d["households"],
                    bedrooms_per_room=d["total_bedrooms"]/d["total_rooms"],
                    #income_per_room=d["median_income"]/d["total_rooms"]
                   )

def new_features(d):
    return d.assign(income_x_age=d["median_income"]*d["housing_median_age"], # Old houses, high income -> possible wealthy area
                    income_per_room=d["median_income"]/d["rooms_per_household"], # income per unit of housing space
                   )

# Lat/long as a smooth 3D position on the sphere, to account for the earth's curvature
def sphere_coords(d):
    lat=np.radians(d["latitude"])
    lon=np.radians(d["longitude"])
    return d.assign(x_coord=np.cos(lat)*np.cos(lon),
                    y_coord=np.cos(lat)*np.sin(lon),
                    z_coord=np.sin(lat))

# Map: expensive houses are near the big cities -> distance to the 2 closest main cities
def dist_city(d):
    dists = cdist(d[["latitude", "longitude"]].to_numpy(), cities)
    sorted_dists = np.sort(dists, axis=1)
    return d.assign(dist_nearest_city=sorted_dists[:, 0],
                    dist_2nd_nearest_city=sorted_dists[:, 1])

# Map: split CA into k-means clusters, and score each district by its similarity (closeness) to each cluster centroid.
# Clusters use coordinates only (no sample weights), so these features contain no price information.
# KM_GAMMA and KM_CLUSTERS were chosen by cross-validation (see the markdown below); they are the default everywhere.
KM_GAMMA, KM_CLUSTERS = 500, 1200

def kmeans_pipeline(gamma=KM_GAMMA, n_clusters=KM_CLUSTERS):
    kmeans = KMeans(n_clusters, n_init=10, random_state=SEED)
    return make_pipeline(kmeans, FunctionTransformer(lambda d: np.exp(-gamma * d ** 2)))

# ---------- EXPERIMENT: cluster price ----------
# CP_CLUSTERS / CP_SMOOTHING are chosen by the grid below (step 1); these values are only placeholders until then.
CP_CLUSTERS, CP_SMOOTHING = 1200, 10

_KMEANS_CACHE = {}  # (n_clusters, coordinates) -> fitted KMeans: every value of m reuses the same k-means (same seed = same result)

def fitted_kmeans(coords, n_clusters):
    key = (n_clusters, hashlib.md5(np.ascontiguousarray(coords).tobytes()).hexdigest())
    if key not in _KMEANS_CACHE:
        _KMEANS_CACHE[key] = KMeans(n_clusters, n_init=10, random_state=SEED).fit(coords)
    return _KMEANS_CACHE[key]

class ClusterPrice(BaseEstimator, TransformerMixin):
    # k-means on (latitude, longitude), then ONE column: the smoothed average training price of the district's cluster.
    # Training rows (fit_transform): out-of-fold, so a district never uses its own price.
    # New rows (transform: early-stopping / validation / test): averages from all training rows.
    def __init__(self, n_clusters=1200, smoothing=10, n_splits=5):
        self.n_clusters, self.smoothing, self.n_splits = n_clusters, smoothing, n_splits

    def _cluster_means(self, labels, y):
        # cluster average pulled towards the overall average; small clusters are pulled more
        sums = np.bincount(labels, weights=y, minlength=self.n_clusters)
        counts = np.bincount(labels, minlength=self.n_clusters)
        return (sums + self.smoothing * y.mean()) / (counts + self.smoothing)

    def fit(self, X, y):
        self.kmeans_ = fitted_kmeans(np.asarray(X, dtype=float), self.n_clusters)   # coordinates only
        self.means_ = self._cluster_means(self.kmeans_.labels_, np.asarray(y, dtype=float))
        return self

    def transform(self, X):
        return self.means_[self.kmeans_.predict(np.asarray(X, dtype=float))].reshape(-1, 1)

    def fit_transform(self, X, y):
        self.fit(X, y)
        y, labels = np.asarray(y, dtype=float), self.kmeans_.labels_
        out = np.empty(len(y))
        for tr, va in KFold(self.n_splits, shuffle=True, random_state=SEED).split(labels):
            out[va] = self._cluster_means(labels[tr], y[tr])[labels[va]]
        return out.reshape(-1, 1)

# Model utilities (same as the report notebook, plus the cluster-price option)

In [ ]:
cv=KFold(10, shuffle=True, random_state=SEED) # same folds for every model -> paired comparisons
scoring={"mse":"neg_mean_squared_error", "mae":"neg_mean_absolute_error", "r2":"r2"}

# steps: row-wise feature functions, applied first (they only use their own row, so no leakage).
# Everything that is fitted (imputer, scalers, ridge) lives inside the pipeline,
# so cross_validate refits it on each training fold only.
# EXPERIMENT: kmeans = False / True (1200 similarity columns) / "price" (1 cluster-price column) / "both"
def location_parts(kmeans, gamma, n_clusters, cp_clusters, cp_smoothing):
    parts = []
    if kmeans in (True, "both"):
        parts.append(("km", kmeans_pipeline(gamma, n_clusters), ["latitude", "longitude"]))
    if kmeans in ("price", "both"):
        # None -> the current CP_CLUSTERS / CP_SMOOTHING (read when the model is built, not when this function is defined)
        cp = ClusterPrice(cp_clusters or CP_CLUSTERS, CP_SMOOTHING if cp_smoothing is None else cp_smoothing)
        parts.append(("price", make_pipeline(cp, StandardScaler()), ["latitude", "longitude"]))
    return parts

def build(steps, model, degree=1, kmeans=False, gamma=KM_GAMMA, n_clusters=KM_CLUSTERS, cp_clusters=None, cp_smoothing=None):
    num_pipeline = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
    cat_pipeline = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    preprocessing = ColumnTransformer(
        [("cat", cat_pipeline, make_column_selector(dtype_exclude="number"))] +
        location_parts(kmeans, gamma, n_clusters, cp_clusters, cp_smoothing),
        remainder=num_pipeline)
    return make_pipeline(*[FunctionTransformer(f) for f in steps], preprocessing, model)

def cross_val_all(models, n_jobs=-1):
    return {name:cross_validate(m, x_train, y_tr_raw, cv=cv, scoring=scoring, return_train_score=True, n_jobs=n_jobs)
            for name, m in models.items()}

# train_* = in-sample, cv_* = out-of-sample (mean over the 10 folds).
# rmse_gain: positive = better than the reference row. folds_better: in how many of the 10 folds it beat the reference.
# Reference = the row above, or the first row if vs_first=True (the first row compares with itself).
def summarise(res, vs_first=False):
    names=list(res)
    mse={n:-res[n]["test_mse"] for n in names}
    out=pd.DataFrame(index=names)
    out["train_rmse"]=[np.sqrt(-res[n]["train_mse"].mean()) for n in names]
    out["train_r2"]=[res[n]["train_r2"].mean() for n in names]
    out["cv_rmse"]=[np.sqrt(mse[n].mean()) for n in names]
    out["cv_mae"]=[-res[n]["test_mae"].mean() for n in names]
    out["cv_r2"]=[res[n]["test_r2"].mean() for n in names]
    ref=[names[0] if vs_first else names[max(i-1, 0)] for i in range(len(names))]
    out["rmse_gain"]=[out.loc[r, "cv_rmse"] - out.loc[n, "cv_rmse"] for n, r in zip(names,ref)]
    out["folds_better"]=[int((mse[n] < mse[r]).sum()) for n, r in zip(names, ref)]
    return out.round(3)

In [ ]:
keep_steps=[ratios, new_features, dist_city, sphere_coords]

# Feature sets: the report's best one, and the two experimental ones
feats = {"engineered_kmeans": (keep_steps, True),          # report: 1200 k-means similarity columns
         "engineered_price": (keep_steps, "price"),        # experiment: 1 cluster-price column instead
         "engineered_kmeans_price": (keep_steps, "both")}  # experiment: both

In [ ]:
# Polynomial terms on all original numeric columns; dummies and k-means distances enter linearly.
# Ridge is included otherwise the error explodes at higher degrees.
# Degree 1 is included as a ridge control, so any gain at degree 2-3 is due to the polynomial terms.
def build_poly(steps, model, degree=1, kmeans=True, gamma=KM_GAMMA, n_clusters=KM_CLUSTERS, cp_clusters=None, cp_smoothing=None):
    original_numeric = ["housing_median_age", "total_rooms", "total_bedrooms", "population",
                    "households", "median_income", "latitude", "longitude"]

    cat_pipeline = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    original_num_pipeline = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                                          PolynomialFeatures(degree, include_bias=False), StandardScaler())
    other_num_pipeline = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
    parts = [("original_num_poly", original_num_pipeline, original_numeric),
             ("cat", cat_pipeline, make_column_selector(dtype_exclude="number"))]
    parts += location_parts(kmeans, gamma, n_clusters, cp_clusters, cp_smoothing)
    preprocessing = ColumnTransformer(parts, remainder=other_num_pipeline)
    return make_pipeline(*[FunctionTransformer(f) for f in steps], preprocessing, model)

ridge=RidgeCV(alphas=np.logspace(-2, 4, 25))

# Neural network

In [ ]:
# Utility functions
def to_tensor(array, dev=None):
    return torch.tensor(np.asarray(array), dtype=torch.float32, device=dev or device)

def parse_architecture(arch_str):
    # converts a string like "128-64" into a tuple (128, 64)
    if arch_str == "linear":
        return ()
    return tuple(int(w) for w in arch_str.split("-"))

# Preprocessing (imputer, scalers, k-means, one-hot) is fitted on the fitting rows ONLY,
# then applied to every other set (early-stopping, validation and test rows)
def prep_fit(steps, X_fit, y_fit, *other_splits, kmeans=False, gamma=KM_GAMMA, n_clusters=KM_CLUSTERS, cp_clusters=None, cp_smoothing=None):
    preprocessor = build(steps, "passthrough", kmeans=kmeans, gamma=gamma, n_clusters=n_clusters,
                         cp_clusters=cp_clusters, cp_smoothing=cp_smoothing)
    X_fit_transformed = preprocessor.fit_transform(X_fit, y_fit)
    other_transformed = [preprocessor.transform(X) for X in other_splits]
    return [X_fit_transformed] + other_transformed

# 1. Construct the network: made it flexible to allow testing for multiple architectures
def build_network(n_inputs, hidden_widths, activation, dropout, use_batchnorm=False):
    # hidden_widths=() gives linear regression
    layers=[]
    in_features=n_inputs

    for width in hidden_widths:
        layers.append(nn.Linear(in_features, width))
        if use_batchnorm:
            layers.append(nn.BatchNorm1d(width))
        layers.append(ACTIVATIONS[activation]())
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        in_features=width

    layers.append(nn.Linear(in_features, 1))
    return nn.Sequential(*layers)

# 2. Train the network
def train_network(X_train, y_train, X_val, y_val, hidden_widths=(128, 64), activation="relu",
                  learning_rate=1e-3, dropout=0.1, weight_decay=1e-4, batch_size=256, use_batchnorm=False,
                  seed=SEED, max_epochs=300, patience=30, device_name=None):
    # Train with MSE loss and AdamW. LR halves on plateau; stops early on validation RMSE and restores the best epoch's weights
    torch.manual_seed(seed)
    dev = torch.device(device_name) if device_name else device # [2-GPU] the GPU this network trains on

    # standardize the target using training statistics only
    y_mean, y_std = y_train.mean(), y_train.std()
    X_train_t = to_tensor(X_train, dev)
    X_val_t = to_tensor(X_val, dev)
    y_train_t = to_tensor((y_train-y_mean)/y_std, dev).unsqueeze(1)

    net = build_network(X_train_t.shape[1], hidden_widths, activation, dropout, use_batchnorm).to(dev)
    optimizer = optim.AdamW(net.parameters(), lr=learning_rate, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=10)
    loss_fn = nn.MSELoss()

    def compute_rmse(X, y_true):
        # RMSE in dollars, with dropout/batchnorm turned off
        net.eval()
        with torch.no_grad():
            preds = net(X).squeeze(1).cpu().numpy() * y_std + y_mean
        return np.sqrt(np.mean((preds - y_true) ** 2))

    best_val_rmse = np.inf
    best_epoch = 0
    best_weights = None
    history = []

    for epoch in range(max_epochs):
        net.train()
        shuffled_idx = torch.randperm(len(X_train_t), device=dev)

        for start in range(0, len(shuffled_idx), batch_size):
            batch_idx = shuffled_idx[start:start + batch_size]
            if len(batch_idx) < 2:
                continue  # batchnorm needs at least 2 rows
            optimizer.zero_grad()
            loss = loss_fn(net(X_train_t[batch_idx]), y_train_t[batch_idx])
            loss.backward()
            optimizer.step()

        train_rmse = compute_rmse(X_train_t, y_train)
        val_rmse = compute_rmse(X_val_t, y_val)
        history.append((train_rmse, val_rmse))
        scheduler.step(val_rmse)

        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_epoch = epoch
            best_weights = {k: v.clone() for k, v in net.state_dict().items()}
        elif epoch - best_epoch >= patience:
            break

    net.load_state_dict(best_weights)
    return {"net": net, "y_mean": y_mean, "y_std": y_std, "best_epoch": best_epoch, "history": history}

# 3. Use the network to make predictions
def predict(result, X):
    result["net"].eval()
    dev = next(result["net"].parameters()).device # [2-GPU] predict on the network's own GPU
    with torch.no_grad():
        preds = result["net"](to_tensor(X, dev)).squeeze(1).cpu().numpy()
    return preds * result["y_std"] + result["y_mean"]

# 4. Tuning
def run_grid(configs, feature_set_name):
    # [2-GPU] same networks as the 1-GPU version, trained in parallel by train_many (one worker per GPU)
    X_train, X_val = data[feature_set_name]
    n_train = len(ya)
    arrays = {"X": np.vstack([X_train, X_val]).astype(np.float32), "y": np.concatenate([ya, yb])}
    fit, stop = np.arange(n_train), np.arange(n_train, n_train + len(yb))
    jobs = [dict(X="X", y="y", fit=fit, stop=stop, predict=[("X", fit), ("X", stop)], cfg=config) for config in configs]
    results_rows = []
    histories = []

    for config, res in zip(configs, train_many(arrays, jobs)):
        pred_train, pred_val = res["preds"]
        arch_label = "-".join(map(str, config["hidden_widths"])) or "linear"
        other_params = {k: v for k, v in config.items() if k != "hidden_widths"}
        results_rows.append({
            "feat": feature_set_name,
            "arch": arch_label,
            **other_params,
            "best_epoch": res["best_epoch"] + 1,
            "train_rmse": np.sqrt(mean_squared_error(ya, pred_train)),
            "val_rmse": np.sqrt(mean_squared_error(yb, pred_val)),
            "val_mae": mean_absolute_error(yb, pred_val),
            "val_r2": r2_score(yb, pred_val),
            "secs": res["secs"],
        })
        histories.append(res["history"])
    return pd.DataFrame(results_rows), histories

# For visualization purposes:
def show_grid(df, rows, cols, value="val_rmse"):
    # Pivot into a rows x cols table of validation RMSE, colored (darker=lower error)
    return df.pivot_table(index=rows, columns=cols, values=value).round(0).style.background_gradient(cmap="viridis_r", axis=None)

def plot_curves(histories, titles):
    # Diagnostic plot for convergence
    n_plots = len(histories)
    n_cols = 4
    n_rows = int(np.ceil(n_plots / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows), sharey=True)
    axes = np.atleast_1d(axes).ravel()

    lowest_val_rmse = min(min(val for _, val in hist) for hist in histories)

    for ax, hist, title in zip(axes, histories, titles):
        hist = np.array(hist)
        ax.plot(hist[:, 0], label="train")
        ax.plot(hist[:, 1], label="validation")
        ax.set_title(title)
        ax.set_ylim(lowest_val_rmse * 0.6, lowest_val_rmse * 2)

    # hide unused subplot slots
    for ax in axes[n_plots:]:
        ax.axis("off")

    axes[0].legend()
    fig.supxlabel("epoch")
    fig.supylabel("RMSE ($)")
    plt.tight_layout()
    plt.show()

In [ ]:
# [2-GPU] Train several networks at the same time: one worker PROCESS per GPU.
# Only the order of the work changes: every network gets exactly the same rows, configuration and seed as in
# housing_part1.ipynb, so the results are the same (up to the usual GPU rounding differences).
# Processes, not threads: each process has its own random-number generators, so a network's seed controls only that network.
# A job = one network: row indices (fit / stop) into a matrix of `arrays`, the configuration (incl. seed),
# and the matrices (or rows) to predict. Large arrays are shared with the workers through files (joblib memmapping).
from joblib import Parallel, delayed

DEVICES = [f"cuda:{i}" for i in range(torch.cuda.device_count())] or ["cpu"]
print("networks are trained on:", DEVICES)

def _train_chunk(arrays, jobs, device_name):
    # runs inside one worker: trains its jobs one after the other on one device
    out = []
    for job in jobs:
        start_time = time.time()
        X, y = arrays[job["X"]], arrays[job["y"]]
        result = train_network(X[job["fit"]], y[job["fit"]], X[job["stop"]], y[job["stop"]], device_name=device_name, **job["cfg"])
        preds = [predict(result, arrays[name] if rows is None else arrays[name][rows]) for name, rows in job["predict"]]
        out.append(dict(preds=preds, best_epoch=result["best_epoch"], history=result["history"], secs=time.time() - start_time))
    return out

def train_many(arrays, jobs):
    # splits the jobs over the devices (job i -> device i % n_devices) and returns one result per job, in the original order
    chunks = [list(range(d, len(jobs), len(DEVICES))) for d in range(len(DEVICES))]
    if len(DEVICES) == 1:
        parts = [_train_chunk(arrays, jobs, DEVICES[0])]
    else:
        parts = Parallel(n_jobs=len(DEVICES))(delayed(_train_chunk)(arrays, [jobs[i] for i in chunk], dev)
                                              for chunk, dev in zip(chunks, DEVICES))
    results = [None] * len(jobs)
    for chunk, part in zip(chunks, parts):
        for i, res in zip(chunk, part):
            results[i] = res
    return results

In [ ]:
# Tuning functions
def to_cfg(row):
    # one row / dict of SPACE values -> keyword arguments for train_network
    return dict(hidden_widths=parse_architecture(row["arch"]), activation=row["activation"],
                learning_rate=float(row["learning_rate"]), dropout=float(row["dropout"]),
                weight_decay=float(row["weight_decay"]), batch_size=int(row["batch_size"]),
                use_batchnorm=bool(row["use_batchnorm"]))

def sample_configs(space, n, seed):
    # same random configurations every time (seeded), so a resumed run continues the same list
    rng = np.random.default_rng(seed)
    seen = set()
    configs = []
    while len(configs) < n:
        c = {k: v[rng.integers(len(v))] for k, v in space.items()}
        key = tuple(c.values())
        if key not in seen:
            seen.add(key)
            configs.append(c)
    return configs

def cached_grid(filename, configs, feat, every=10):
    # run_grid with a CSV checkpoint every `every` networks; resumes after a Colab disconnect
    path = os.path.join(OUTPUT_DIR, filename)
    done = pd.read_csv(path) if os.path.exists(path) else pd.DataFrame()
    for i in range(len(done), len(configs), every):
        part, _ = run_grid(configs[i:i + every], feat)
        done = pd.concat([done, part], ignore_index=True)
        done.to_csv(path, index=False)
    return done

def screen_survivors(screen, space, keep):
    # best `keep[dim]` values of each hyperparameter, ranked by mean validation RMSE
    survivors = {}
    for dim in space:
        stats = screen.groupby(dim)["val_rmse"].agg(mean="mean", median="median", best="min", n="count")
        stats = stats.sort_values("median")
        display(dim, stats.round(0))
        survivors[dim] = list(stats.index[:keep[dim]])
    return survivors

def recheck_top(final, feat, n_recheck, space, seeds):
    # re-run the best `n_recheck` networks with new seeds, since a single run can be noisy
    rows = []
    for _, row in final.sort_values("val_rmse").head(n_recheck).iterrows():
        repeats, _ = run_grid([{**to_cfg(row), "seed": s} for s in seeds], feat)
        rows.append(dict(
            **{k: row[k] for k in space},
            step2_mean=row["val_rmse"],
            mean_seeds=repeats["val_rmse"].mean(),
            sd_seeds=repeats["val_rmse"].std(),
            mean_best_epoch=repeats["best_epoch"].mean(),
        ))
    return pd.DataFrame(rows).sort_values("mean_seeds")

def tune_nn(feat, space, n_screen, keep, max_final, n_recheck, recheck_seeds, seed, tag, step2_seeds):
    # step 1: broad random screen over the whole space
    screen = cached_grid(f"nn_screen_{tag}.csv", [to_cfg(c) for c in sample_configs(space, n_screen, seed)], feat)
    survivors = screen_survivors(screen, space, keep)
    print("surviving values:", survivors)

    # step 2: full grid over the survivors only (subsampled if larger than max_final)
    combos = list(itertools.product(*survivors.values()))
    if len(combos) > max_final:
        pick = np.random.default_rng(seed).choice(len(combos), max_final, replace=False)
        combos = [combos[i] for i in np.sort(pick)]

    configs=[to_cfg(dict(zip(survivors, c))) for c in combos]
    runs=[cached_grid(f"nn_final_{tag}_seed{s}.csv", [{**c, "seed":s} for c in configs], feat) for s in step2_seeds]
    # one row per configuration, every result column averaged over the step-2 seeds
    metric_cols=["best_epoch", "train_rmse", "val_rmse", "val_mae", "val_r2", "secs"]
    final=pd.concat(runs).groupby(list(space), as_index=False, sort=False)[metric_cols].mean()

    display(final.sort_values("val_rmse").head(15).round(3))
    display(show_grid(final, "arch", ["activation", "learning_rate"]))
    display(show_grid(final, "dropout", ["weight_decay", "batch_size"]))

    # step 3: re-run the best few with new seeds, since a single run is noisy
    recheck = recheck_top(final, feat, n_recheck, space, recheck_seeds)
    display(recheck.round(3))
    return to_cfg(recheck.iloc[0]), screen, final, recheck

In [ ]:
def cross_validate_nn(cfg, feat, seeds=(89, 233, 1597), **prep_kwargs):
    # Honest CV: each fold trains a fresh ensemble of `seeds` networks, with preprocessing fit on that
    # fold's training rows only. Early stopping watches a held-out 10% split (`stop`), never the
    # validation fold (`val`), so nothing about the validation fold leaks into training decisions.
    # Keys match cross_validate's naming (train_mse/test_mse/...) so summarise() works on both.
    # [2-GPU] the preprocessing of all folds is done first, then all 10 x 3 networks are trained in parallel.
    steps, km = feats[feat]
    y = y_tr_raw
    fold_scores = dict(train_mse=[], test_mse=[], test_mae=[], train_r2=[], test_r2=[])
    test_preds = []
    oof_preds = np.zeros(len(y))

    arrays, jobs, folds = {}, [], []
    for f, (train_idx, val_idx) in enumerate(tqdm(list(cv.split(x_train)), desc="preprocessing folds")):
        fit_idx, stop_idx = train_test_split(train_idx, test_size=0.1, random_state=SEED)
        X_fit, X_stop, X_val, X_test = prep_fit(
            steps, x_train.iloc[fit_idx], y[fit_idx], x_train.iloc[stop_idx],
            x_train.iloc[val_idx], x_test, kmeans=km, **prep_kwargs) # EXPERIMENT: prep_kwargs = cp_clusters / cp_smoothing
        # rows of fold f stacked as [fit | stop | val]; the test set in its own matrix
        arrays[f"X{f}"] = np.vstack([X_fit, X_stop, X_val]).astype(np.float32)
        arrays[f"y{f}"] = np.concatenate([y[fit_idx], y[stop_idx], y[val_idx]])
        arrays[f"T{f}"] = np.asarray(X_test, dtype=np.float32)
        n_fit, n_stop = len(fit_idx), len(stop_idx)
        r_fit, r_stop, r_val = np.arange(n_fit), np.arange(n_fit, n_fit + n_stop), np.arange(n_fit + n_stop, n_fit + n_stop + len(val_idx))
        for seed in seeds:
            jobs.append(dict(X=f"X{f}", y=f"y{f}", fit=r_fit, stop=r_stop,
                             predict=[(f"X{f}", r_fit), (f"X{f}", r_val), (f"T{f}", None)], cfg={**cfg, "seed": seed}))
        folds.append((fit_idx, val_idx))

    results = train_many(arrays, jobs)

    for f, (fit_idx, val_idx) in enumerate(folds):
        fold_results = results[f * len(seeds):(f + 1) * len(seeds)]
        fit_pred = np.mean([r["preds"][0] for r in fold_results], axis=0)  # ensemble = average of the seeds
        val_pred = np.mean([r["preds"][1] for r in fold_results], axis=0)
        test_preds.append(np.mean([r["preds"][2] for r in fold_results], axis=0))
        oof_preds[val_idx] = val_pred

        fold_scores["train_mse"].append(-mean_squared_error(y[fit_idx], fit_pred))
        fold_scores["test_mse"].append(-mean_squared_error(y[val_idx], val_pred))
        fold_scores["test_mae"].append(-mean_absolute_error(y[val_idx], val_pred))
        fold_scores["train_r2"].append(r2_score(y[fit_idx], fit_pred))
        fold_scores["test_r2"].append(r2_score(y[val_idx], val_pred))

    fold_scores = {k: np.array(v) for k, v in fold_scores.items()}
    return fold_scores, np.mean(test_preds, axis=0), oof_preds

In [ ]:
# Features to test
ACTIVATIONS = {"relu":nn.ReLU, "leaky_relu":nn.LeakyReLU, "elu":nn.ELU, "silu":nn.SiLU, "gelu":nn.GELU, "tanh":nn.Tanh}
FINAL_FEATURE_SET = "engineered_price" # EXPERIMENT: set automatically in step 1

# Tuning configuration
"""
Step 1: random search over everything at once; for each value of each hyperparameter we look at the average
  validation RMSE of all the networks that used it. Therefore, a value that is consistently worse is dropped.
Step 2: full search over the "surviving" values (if < MAX_FINAL, otherwise another random search).
Step 3: the best networks are re-run with new seeds and the best average is chosen as the final configuration.
Both steps are saved to CSV in OUTPUT_DIR and resume after a Colab disconnect; delete the CSV files to repeat a step.
"""

# EXPERIMENT configurations:
# REF_CFG   = the report's final NN configuration (tuned for the 1200 k-means similarity columns): used for the comparison in step 4
# START_CFG = configuration used in the step-1 grid. It was tuned with k=75 (~96 inputs), closer to the ~23 inputs of the
#             cluster-price set than REF_CFG (tuned for ~1221 inputs). The full retuning in step 2 replaces it.
REF_CFG = dict(hidden_widths=(256,128,64), activation="leaky_relu", learning_rate=0.0005, dropout=0.2,
               weight_decay=0.0, batch_size=256, use_batchnorm=True)
START_CFG = dict(hidden_widths=(256,128), activation="relu", learning_rate=0.005, dropout=0.2,
                 weight_decay=0.001, batch_size=128, use_batchnorm=True)
FIXED_CFG = START_CFG # used only if RUN_TUNING = False
RUN_TUNING = True # False: skip the search and use FIXED_CFG
N_SCREEN = 180 # Networks to test in step 1
KEEP = dict(arch=3, activation=3, learning_rate=2, dropout=2,
            weight_decay=1, batch_size=1, use_batchnorm=1) # How many values of each hyperparameter survive step 1
MAX_FINAL = 120 # Maximum number of networks to test in step 2
N_RECHECK = 5 # Top networks from step 2, to test with new seeds at the end to eliminate noise
RECHECK_SEEDS = (9, 28, 630, 2026, 40121, 271828, 7778777)
STEP2_SEEDS=(31337, 65537, 104729)

# Hyperparameter candidates
SPACE = dict(
    arch=["256", "64-64", "128-64", "256-128", "128-64-32", "256-128-64"],
    activation=["relu", "leaky_relu", "elu", "silu", "gelu", "tanh"],
    learning_rate=[1e-4, 5e-4, 1e-3, 5e-3, 1e-2],
    dropout=[0, 0.05, 0.1, 0.2, 0.3, 0.4],
    weight_decay=[0, 1e-3, 1e-2, 1e-1, 1],
    batch_size=[64, 128, 256, 512],
    use_batchnorm=[False, True],
)

# Step 1: tuning `n_clusters` × `m`

Same 10 CV folds as everywhere else. `n_clusters` ∈ {75, 300, 600, 1200, 2000, 3000, 5000} × `m` ∈ {1, 3, 10, 30, 100} (35 pairs).

* **Ridge** (degree 1, engineered + cluster price) on the full grid: cheap, reported for context only.
* **NN** (`START_CFG`, 3-seed ensemble per fold, 30 networks per pair) on the full grid: **this decides the pair**.
* Then one extra NN row at the chosen pair: **cluster price + the 1200 similarity columns** ("both"), to see whether the price column replaces the similarities or adds to them.

Results are saved to `OUTPUT_DIR/cluster_price_tests/` after every pair; re-running a cell skips the pairs already done. In the summary tables, `rmse_gain` / `folds_better` compare every pair with the **best** pair, so pairs within noise of the best are easy to spot.

In [ ]:
CP_TEST_DIR = os.path.join(OUTPUT_DIR, "cluster_price_tests")
os.makedirs(CP_TEST_DIR, exist_ok=True)
METRICS = ["train_mse", "test_mse", "test_mae", "train_r2", "test_r2"]
CP_CLUSTERS_GRID = [75, 300, 600, 1200, 2000, 3000, 5000]
CP_M_GRID = [1, 3, 10, 30, 100]

def run_pair_grid(filename, pairs, score_fn):
    # score_fn(n_clusters, m) -> per-fold scores in cross_validate format. One CSV row per fold, saved after every pair
    path = os.path.join(CP_TEST_DIR, filename)
    done = pd.read_csv(path) if os.path.exists(path) else None
    for k, m in pairs:
        if done is not None and ((done["n_clusters"] == k) & (done["m"] == m)).any():
            continue
        start_time = time.time()
        scores = score_fn(k, m)
        rows = pd.DataFrame({"n_clusters": k, "m": m, "fold": range(len(scores["test_mse"])),
                             **{s: scores[s] for s in METRICS}})
        done = rows if done is None else pd.concat([done, rows], ignore_index=True)
        done.to_csv(path, index=False)
        print(f"n_clusters={k}, m={m}: cv_rmse={np.sqrt(-np.mean(scores['test_mse'])):,.0f} ({time.time() - start_time:.0f}s)")
    return done

def pair_summary(df):
    # summarise() table, one row per (n_clusters, m), best first; rmse_gain / folds_better are measured against the best pair
    pairs = list(dict.fromkeys(zip(df["n_clusters"], df["m"])))
    res = {}
    for k, m in pairs:
        d = df[(df["n_clusters"] == k) & (df["m"] == m)].sort_values("fold")
        res[(k, m)] = {s: d[s].to_numpy() for s in METRICS}
    best = min(res, key=lambda p: -res[p]["test_mse"].mean())
    order = [best] + [p for p in pairs if p != best]
    out = summarise({f"k={k}, m={m:g}": res[(k, m)] for k, m in order}, vs_first=True)
    out.insert(0, "n_clusters", [k for k, _ in order])
    out.insert(1, "m", [m for _, m in order])
    return out.sort_values("cv_rmse")

def pair_heatmap(summary, value="cv_rmse"):
    # n_clusters x m table (darker = better)
    table = summary.pivot_table(index="n_clusters", columns="m", values=value)
    table.columns = [f"m={m:g}" for m in table.columns]
    return table.round(0).style.background_gradient(cmap="viridis_r", axis=None)

In [ ]:
# ---------- ridge on the full grid (context) ----------
# n_jobs=1: k-means already uses all CPU cores, and running in this process lets every m reuse the cached k-means
def score_ridge(k, m):
    model = build_poly(keep_steps, ridge, degree=1, kmeans="price", cp_clusters=k, cp_smoothing=m)
    return cross_validate(model, x_train, y_tr_raw, cv=cv, scoring=scoring, return_train_score=True, n_jobs=1)

ridge_cp = run_pair_grid("ridge_folds.csv", itertools.product(CP_CLUSTERS_GRID, CP_M_GRID), score_ridge)
ridge_cp_summary = pair_summary(ridge_cp)
ridge_cp_summary.to_csv(os.path.join(CP_TEST_DIR, "ridge_summary.csv"))
display(ridge_cp_summary)
display(pair_heatmap(ridge_cp_summary))

In [ ]:
# ---------- NN on the full grid (decides the pair) ----------
def score_nn(k, m):
    return cross_validate_nn(START_CFG, "engineered_price", cp_clusters=k, cp_smoothing=m)[0]

nn_cp = run_pair_grid("nn_folds.csv", itertools.product(CP_CLUSTERS_GRID, CP_M_GRID), score_nn)
nn_cp_summary = pair_summary(nn_cp)
nn_cp_summary.to_csv(os.path.join(CP_TEST_DIR, "nn_summary.csv"))
display(nn_cp_summary)
display(pair_heatmap(nn_cp_summary))

In [ ]:
# ---------- choice of the pair and of the feature set ----------
CHOSEN_PAIR = None # EDIT: (n_clusters, m) to override the automatic choice (lowest NN CV RMSE)
def as_number(x):
    # 3.0 -> 3 (plain Python number, for readable prints and file names)
    return int(x) if float(x).is_integer() else float(x)

CP_CLUSTERS, CP_SMOOTHING = CHOSEN_PAIR or (int(nn_cp_summary.iloc[0]["n_clusters"]), as_number(nn_cp_summary.iloc[0]["m"]))
print(f"chosen pair: n_clusters={CP_CLUSTERS}, m={CP_SMOOTHING:g}")

# "both" = cluster price + the 1200 k-means similarity columns, at the chosen pair
nn_both = run_pair_grid("nn_both_folds.csv", [(CP_CLUSTERS, CP_SMOOTHING)],
                        lambda k, m: cross_validate_nn(START_CFG, "engineered_kmeans_price", cp_clusters=k, cp_smoothing=m)[0])
price_only = nn_cp[(nn_cp["n_clusters"] == CP_CLUSTERS) & (nn_cp["m"] == CP_SMOOTHING)].sort_values("fold")
both = nn_both[(nn_both["n_clusters"] == CP_CLUSTERS) & (nn_both["m"] == CP_SMOOTHING)].sort_values("fold")
fs_cmp = summarise({"engineered_price": {s: price_only[s].to_numpy() for s in METRICS},
                    "engineered_kmeans_price": {s: both[s].to_numpy() for s in METRICS}}, vs_first=True)
display(fs_cmp)

CHOSEN_FEATURE_SET = None # EDIT: "engineered_price" or "engineered_kmeans_price" to override the automatic choice
FINAL_FEATURE_SET = CHOSEN_FEATURE_SET or fs_cmp["cv_rmse"].idxmin()
print("feature set for the rest of the notebook:", FINAL_FEATURE_SET)

# Step 2: NN retuning on the chosen feature set

Same procedure as the report (random screen → full grid over the surviving values → recheck of the best configurations with new seeds), on an 80/20 split, with the chosen `n_clusters` / `m`. The tuning CSVs are tagged with the feature set and the pair, so they never mix with other runs.

In [ ]:
# Divide into train/validation split for tuning (80/20, random)
idx_tr, idx_va = train_test_split(np.arange(len(y_tr_raw)), test_size=0.2, random_state=SEED)
xa, ya = x_train.iloc[idx_tr], y_tr_raw[idx_tr]
xb, yb = x_train.iloc[idx_va], y_tr_raw[idx_va]

steps, km = feats[FINAL_FEATURE_SET]
data = {FINAL_FEATURE_SET: tuple(prep_fit(steps, xa, ya, xb, kmeans=km))} # EXPERIMENT: only the chosen feature set
print({name: d[0].shape[1] for name, d in data.items()}) # number of input columns per feature set

# reference: the linear model (best features) on this exact split
Xa, Xb = data[FINAL_FEATURE_SET]
lm = LinearRegression().fit(Xa, ya)
print("linear model on this split | train RMSE:", round(np.sqrt(mean_squared_error(ya, lm.predict(Xa)))),
      "| validation RMSE:", round(np.sqrt(mean_squared_error(yb, lm.predict(Xb)))))


In [ ]:
# Tuning
if RUN_TUNING:
    best_cfg, screen, final, recheck = tune_nn(
        feat=FINAL_FEATURE_SET, space=SPACE, n_screen=N_SCREEN, keep=KEEP, max_final=MAX_FINAL,
        n_recheck=N_RECHECK, recheck_seeds=RECHECK_SEEDS, seed=SEED, tag=f"EXP_{FINAL_FEATURE_SET}_k{CP_CLUSTERS}_m{CP_SMOOTHING:g}", step2_seeds=STEP2_SEEDS)
else:
    best_cfg = FIXED_CFG

print(best_cfg)

# Step 3: recheck of the top pairs with the retuned configuration

The step-1 grid used `START_CFG`; the pair was then kept fixed during the retuning. Here the 3 best pairs of the NN grid are cross-validated again with the **retuned** configuration, to check that the choice does not depend on the starting configuration. If another pair clearly wins here, set `CHOSEN_PAIR` in step 1 to it and re-run from step 1 (everything already computed is loaded from the CSV files).

In [ ]:
top_pairs = [(int(k), as_number(m)) for k, m in nn_cp_summary[["n_clusters", "m"]].head(3).to_numpy()]
recheck_cp = run_pair_grid(f"nn_recheck_EXP_{FINAL_FEATURE_SET}.csv", top_pairs,
                           lambda k, m: cross_validate_nn(best_cfg, FINAL_FEATURE_SET, cp_clusters=k, cp_smoothing=m)[0])
recheck_summary = pair_summary(recheck_cp)
display(recheck_summary)
best_recheck = (int(recheck_summary.iloc[0]["n_clusters"]), as_number(recheck_summary.iloc[0]["m"]))
print("best pair with the retuned configuration:", best_recheck,
      "| same as the chosen pair" if best_recheck == (CP_CLUSTERS, CP_SMOOTHING) else "| DIFFERENT from the chosen pair: see the note above")

# Step 4: comparison with the current best model, and submissions

Same 10 folds, every row compared with the first one (the report's final NN: k-means 1200/500 similarity columns, `REF_CFG`).

In [ ]:
exp_cv, exp_test, exp_oof = cross_validate_nn(best_cfg, FINAL_FEATURE_SET)
ref_cv, _, _ = cross_validate_nn(REF_CFG, "engineered_kmeans")
ridge_exp = cross_val_all({"ridge deg1 | engineered_price": build_poly(keep_steps, ridge, 1, kmeans="price")}, n_jobs=1)
display(summarise({"NN | engineered_kmeans (report, current best)": ref_cv, **ridge_exp,
                   f"NN | {FINAL_FEATURE_SET} (experiment)": exp_cv}, vs_first=True))

def save_submission(test_pred, filename):
    print(filename, "| non-positive predictions before clipping:", (test_pred <= 0).sum(),
          "| above the training max:", (test_pred > y_tr_raw.max()).sum())
    pred = np.clip(test_pred, y_tr_raw.min(), y_tr_raw.max())
    pd.DataFrame({"ID": x_test_df.iloc[:, 0], "Price": pred}).to_csv(os.path.join(OUTPUT_DIR, filename), index=False)

# test predictions of the 30 CV networks above (10 folds x 3 seeds)
save_submission(exp_test, "submission_EXP_cluster_price_cv.csv")

In [ ]:
# Final NN: preprocessing fitted on ALL training rows. 10 groups x 3 seeds = 30 networks averaged.
# Each group learns from 90% of the rows and early-stops on the other 10%; the 10% rotates (like the CV folds),
# so every row is used for learning and every network still stops at its best epoch on rows it did not learn from.
# [2-GPU] the 30 networks are trained in parallel.
steps, km = feats[FINAL_FEATURE_SET]
X_all, X_test_all = prep_fit(steps, x_train, y_tr_raw, x_test, kmeans=km)

arrays = {"X": np.asarray(X_all, dtype=np.float32), "y": y_tr_raw, "T": np.asarray(X_test_all, dtype=np.float32)}
jobs = [dict(X="X", y="y", fit=fit_idx, stop=stop_idx, predict=[("T", None)], cfg={**best_cfg, "seed": seed})
        for fit_idx, stop_idx in KFold(10, shuffle=True, random_state=SEED).split(X_all)
        for seed in (89, 233, 1597)]
test_preds = [res["preds"][0] for res in train_many(arrays, jobs)]

save_submission(np.mean(test_preds, axis=0), "submission_EXP_cluster_price.csv")